In [18]:
import numpy as np
import matplotlib.pyplot as plt
import torchvision
import torch.nn.functional as F

In [19]:
data = torchvision.datasets.MNIST(root='../data', download=True, train=True,
                                        transform=torchvision.transforms.ToTensor())
train_data =data.data.numpy()

train_out = F.one_hot(data.train_labels, 10).numpy()


In [20]:
class ConvNet:
    def __init__(self, layers, pool_size=2):
        self.layers_tapes = []
        self.layers = layers
        self.weights = []
        self.biases = []
        for layer in layers:
            if len(layer) == 2:
                self.layers_tapes.append('lin')
                self.generate_params_for_linear_layer(layer[0], layer[1])
            else:
                self.layers_tapes.append('conv')
                self.generate_kernels(layer[1], layer[2])
        self.pool_size = pool_size
        self.softmax_T = 1
        self.lr = 0.01
        self.iteration = 0

    def generate_params_for_linear_layer(self, input_size, output_size):
        self.weights.append(np.random.normal(loc=0, scale=1, size=(output_size, input_size)) / np.sqrt(input_size))
        self.biases.append(np.random.normal(size=output_size).reshape(-1, 1))

    def generate_kernels(self, length, kernel_size):
        weights = []
        biases = []
        for _ in range(length):
            z = 1
            if len(self.weights) != 0:
                z = self.weights[-1].shape[0]
            weights.append(np.random.randn(z, kernel_size, kernel_size) / kernel_size)
            biases.append(0.0)
        self.weights.append(np.array(weights))
        self.biases.append(np.array(biases))

    def create_feature_map(self, input, length, weights, biases):
        feature_maps = []
        kernel_width = weights.shape[2]
        kernel_height = weights.shape[3]

        input_height = input.shape[1] - kernel_height + 1
        input_width = input.shape[2] - kernel_width + 1

        for i in range(length):
            feature_map = np.zeros((input_height, input_width))
            for y in range(input_height):
                for x in range(input_width):
                    feature_map[y, x] = np.sum(input[:, y:y + kernel_height, x:x + kernel_width] * weights[i]) + biases[
                        i]
            feature_maps.append(feature_map)
        return feature_maps

    def relu(self, feature_maps):
        output = []
        for feature_map in feature_maps:
            output.append(np.maximum(0, feature_map))
        self.activations.append(np.array(output))
        return output

    def max_pool(self, feature_maps):
        output = []
        out_height, out_width = feature_maps[0].shape[0] // self.pool_size, feature_maps[0].shape[1] // self.pool_size
        for feature_map in feature_maps:
            out = np.zeros((out_height, out_width))
            for y in range(out_height):
                for x in range(out_width):
                    start_y = y * self.pool_size
                    start_x = x * self.pool_size
                    out[y, x] = np.max(feature_map[start_y:start_y + self.pool_size, start_x:start_x + self.pool_size])
            output.append(out)
        self.activations.append(np.array(output))

        '''self.iteration += 1
        if self.iteration == 8000 or self.iteration == 8001 or self.iteration ==8002:
            self.show([*feature_maps])
            self.show([*output])
'''
        return output

    def forward(self, x):
        self.activations = [x]
        self.pipeline = []
        iter = 0
        for layer, layer_type in zip(self.layers, self.layers_tapes):
            if layer_type == 'conv':
                self.pipeline.extend(['conv', 'pool'])
                feature_maps = self.create_feature_map(x, layer[1], self.weights[iter], self.biases[iter])
                x = np.array(self.max_pool(self.relu(feature_maps)))
            if layer_type == 'lin':
                if 'flat' not in self.pipeline:
                    self.pipeline.append('flat')
                self.pipeline.append('lin')
                x = x.reshape(-1, 1)
                self.activations.append(x)
                z = self.weights[iter].dot(x) + self.biases[iter]
                if iter == len(self.layers) - 1:
                    z_shifted = z - np.max(z, axis=0, keepdims=True)
                    S = np.sum(np.exp(self.softmax_T * z_shifted), axis=0)
                    self.activations.append(np.exp(self.softmax_T * z_shifted) / S)
                else:
                    x = np.maximum(0, z)
            iter += 1
        self.pipeline.append('out')
        return self.activations[-1]

    def bp1(self, a, y):
        self.layers_mistakes = []
        y = y.reshape(-1, 1)
        self.layers_mistakes.append(a - y)

    def bp2(self, i, layer_id):
        relu_derivative = np.where(self.activations[-i - 1] > 0, 1, 0).reshape(self.activations[-i - 1].shape)
        self.layers_mistakes.append(self.weights[layer_id].T.dot(self.layers_mistakes[-1]) * relu_derivative)
        self.current_b_grads[layer_id] = self.layers_mistakes[-2]
        self.current_w_grads[layer_id] = self.layers_mistakes[-2].dot(self.activations[-i - 1].T)

    def weights_back_conv(self, i, layer_id):
        in_img = self.activations[-i - 1]
        out = self.layers_mistakes[-1]
        kernel_size = in_img.shape[1] - out.shape[1] + 1
        kernel_shape = (out.shape[0], in_img.shape[0], kernel_size, kernel_size)
        kernel_mistake = np.zeros(kernel_shape)
        for x in range(kernel_shape[0]):
            for y in range(kernel_shape[1]):
                for k in range(kernel_size):
                    for j in range(kernel_size):
                        kernel_mistake[x][y][k][j] = np.sum(in_img[y][k:k + out.shape[1], j:j + out.shape[2]] * out[x])

        self.current_w_grads[layer_id] = kernel_mistake / (out.shape[0] * out.shape[1] * out.shape[2])

    def flatten_back(self, i):
        self.layers_mistakes.append(self.layers_mistakes[-1].reshape(self.activations[-i - 1].shape))

    def pool_mistake(self, i):
        repeated_error = np.repeat(self.layers_mistakes[-1], self.pool_size, axis=1)
        repeated_error = np.repeat(repeated_error, self.pool_size, axis=2)
        repeated_act = np.repeat(self.activations[-i], self.pool_size, axis=1)
        repeated_act = np.repeat(repeated_act, self.pool_size, axis=2)
        mask = np.where(self.activations[-i - 1] == repeated_act, 1, 0)
        self.layers_mistakes.append(repeated_error * mask)

    def conv_mistake(self, i, layer_id):
        conv_bias_grad = np.sum(self.layers_mistakes[-1], axis=(1, 2))
        self.current_b_grads[layer_id] = conv_bias_grad.reshape(self.biases[layer_id].shape)

        relu_derivative = np.where(self.activations[-i - 1] > 0, 1, 0).reshape(self.activations[-i - 1].shape)
        kernel_size = self.weights[layer_id].shape[2]
        num_channel = self.weights[layer_id].shape[1]
        back_kernel = np.flip(self.weights[layer_id], axis=(-2, -1))
        pad_size = kernel_size - 1
        mistakes_pd = np.pad(self.layers_mistakes[-1], ((0, 0), (pad_size, pad_size), (pad_size, pad_size)))
        layer_mistake = np.zeros(self.activations[-i - 1].shape)
        for z in range(num_channel):
            for x in range(layer_mistake.shape[1]):
                for y in range(layer_mistake.shape[2]):
                    layer_mistake[z, x, y] = np.sum(
                        mistakes_pd[:, x:x + kernel_size, y:y + kernel_size] * back_kernel[:, z, :, :])
        self.layers_mistakes.append(layer_mistake * relu_derivative)

    def fit(self, x, y, batch_size=10):
        iterations = y.shape[0] // batch_size
        for chunk in range(iterations):
            weights_batch = [np.zeros_like(w) for w in self.weights]
            biases_batch = [np.zeros_like(b) for b in self.biases]
            start_i = chunk * batch_size
            for j in range(batch_size):
                self.current_w_grads = [np.zeros_like(w) for w in self.weights]
                self.current_b_grads = [np.zeros_like(b) for b in self.biases]
                if len(x[start_i + j].shape) == 2:
                    a = np.expand_dims(x[start_i + j], axis=0)
                    y_net = self.forward(a)
                else:
                    y_net = self.forward(x[start_i + j])
                self.bp1(y_net, y[start_i + j])
                pipeline = self.pipeline[::-1]
                current_layer = len(self.weights) - 1
                for i in range(1, len(pipeline)):
                    if pipeline[i] == 'lin':
                        self.bp2(i, current_layer)
                        current_layer -= 1
                    if pipeline[i] == 'flat':
                        self.flatten_back(i)
                    if pipeline[i] == 'pool':
                        self.pool_mistake(i)
                    if pipeline[i] == 'conv':
                        self.weights_back_conv(i, current_layer)
                        if current_layer == 0:
                            conv_bias_grad = np.mean(self.layers_mistakes[-1], axis=(1, 2))
                            self.current_b_grads[current_layer] = conv_bias_grad.reshape(
                                self.biases[current_layer].shape)
                        else:
                            self.conv_mistake(i, current_layer)
                        current_layer -= 1

                for i in range(len(self.weights)):
                    weights_batch[i] += self.current_w_grads[i]
                for i in range(len(self.biases)):
                    biases_batch[i] += self.current_b_grads[i]

            for k in range(len(self.weights)):
                self.weights[k] -= self.lr * weights_batch[k] / batch_size
            for k in range(len(self.biases)):
                self.biases[k] -= self.lr * biases_batch[k] / batch_size

    def show(self, images):
        count = len(images)
        if count == 1:
            plt.imshow(images[0], cmap='gray')
        else:
            fig, axs = plt.subplots(nrows=1, ncols=count)
            for i, img in enumerate(images):
                axs[i].imshow(img, cmap='gray')

    def loss(self, x, y):
        if len(x.shape) == 2:
            a = np.expand_dims(x, axis=0)
            y_net = self.forward(a)
        else:
            y_net = self.forward(x)

        prob = np.clip(y_net[np.argmax(y)], 1e-15, 1.0)
        return -np.log(prob)



net1 = ConvNet(([28*28, 3, 3], [13*13, 2, 2], [72, 10]))
net2 = ConvNet(([28 * 28, 3, 3], [507, 10]))

net1.fit(train_data[0:2000], train_out[0:2000], 20)
net2.fit(train_data[0:2000], train_out[0:2000], 20)




In [21]:
for i in [1, 2, 3, 4, 5, 6, 7, 8]:
    print('net1 - net2 - expected:')
    print(np.argmax(net1.forward(np.array([train_data[i]]))), "-", np.argmax(net2.forward(np.array([train_data[i]]))), "-", np.argmax(train_out[i]))
    print()
    print('loss value(net1 - net2):')
    print(net1.loss(np.array([train_data[i]]), train_out[i]), "-", net2.loss(np.array([train_data[i]]), train_out[i]))
    print()


net1 - net2 - expected:
0 - 0 - 0

loss value(net1 - net2):
[-0.] - [-0.]

net1 - net2 - expected:
4 - 4 - 4

loss value(net1 - net2):
[0.02071323] - [-0.]

net1 - net2 - expected:
1 - 1 - 1

loss value(net1 - net2):
[-0.] - [-0.]

net1 - net2 - expected:
7 - 9 - 9

loss value(net1 - net2):
[20.10910441] - [-0.]

net1 - net2 - expected:
2 - 2 - 2

loss value(net1 - net2):
[-0.] - [-0.]

net1 - net2 - expected:
1 - 1 - 1

loss value(net1 - net2):
[7.97035363e-09] - [-0.]

net1 - net2 - expected:
3 - 3 - 3

loss value(net1 - net2):
[-0.] - [-0.]

net1 - net2 - expected:
1 - 1 - 1

loss value(net1 - net2):
[0.01245965] - [-0.]

